# LeetCode 3151 — Special Array I bằng Python

Link đề: https://leetcode.com/problems/special-array-i/ · Độ khó: **Easy** · Chủ đề: Array (mảng), chẵn/lẻ (parity)

Skeleton LeetCode:

```python
class Solution:
    def isArraySpecial(self, nums: List[int]) -> bool:
        ...
```

Ghi chú: đề và các ví dụ được viết lại theo đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một mảng là **đặc biệt (special)** khi **mọi cặp phần tử đứng cạnh nhau** có **tính chẵn/lẻ (parity) khác nhau** — một số chẵn, một số lẻ.
2. Nắm các công cụ Python: toán tử **`%`** (chia lấy dư) để biết chẵn/lẻ, **`range(len(nums) - 1)`** để duyệt các cặp kề nhau, **`zip(nums, nums[1:])`**, **`all(...)`**, và mẹo bit **`x & 1`**, **`a ^ b`**.
3. Đi từ cách **xét mọi cặp rồi lọc cặp kề nhau** (chậm, dễ hiểu) → **một vòng lặp so sánh cặp kề nhau** → **`zip` + `all`** → **mẹo bit** → **mẫu chẵn/lẻ xen kẽ** (đoán trước dãy chẵn lẻ phải như thế nào).
4. Hiểu mấu chốt: mảng chỉ đặc biệt khi **không có cặp kề nhau nào cùng chẵn lẻ**; chỉ cần **tìm một cặp vi phạm** là kết luận `false` ngay (dừng sớm), còn nếu duyệt hết mà không thấy thì `true`.
5. Tránh các lỗi kinh điển: duyệt `range(len(nums))` rồi đọc `nums[i + 1]` (**IndexError**), **`return True` quá sớm** khi mới thấy một cặp tốt, so sánh **giá trị** thay vì **chẵn/lẻ**, và quên mảng **một phần tử** luôn là `true`.

## 2. Đề bài & ví dụ

Một mảng được gọi là **đặc biệt** nếu **tính chẵn/lẻ của mọi cặp phần tử kề nhau đều khác nhau**. Nói cách khác, trong mỗi cặp kề nhau, **một phần tử phải chẵn và phần tử còn lại phải lẻ**.

Cho mảng số nguyên `nums`. Trả về `true` nếu `nums` là mảng đặc biệt, ngược lại trả về `false`.

**Ví dụ 1**

- Input: `nums = [1]`
- Output: `true`
- Giải thích: chỉ có một phần tử, không có cặp nào để vi phạm, nên là `true`.

**Ví dụ 2**

- Input: `nums = [2,1,4]`
- Output: `true`
- Giải thích: có hai cặp kề nhau: `(2,1)` và `(1,4)`. Cả hai cặp đều gồm một số chẵn và một số lẻ.

**Ví dụ 3**

- Input: `nums = [4,3,1,6]`
- Output: `false`
- Giải thích: `nums[1] = 3` và `nums[2] = 1` đều là số lẻ, nên cặp `(3,1)` vi phạm.

**Ràng buộc:** `1 <= nums.length <= 100`, `1 <= nums[i] <= 100`.

In [1]:
from typing import List
import random
import itertools

examples = [
    ([1], True),
    ([2, 1, 4], True),
    ([4, 3, 1, 6], False),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[1] -> kỳ vọng True
[2, 1, 4] -> kỳ vọng True
[4, 3, 1, 6] -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. Chẵn / lẻ và phép chia lấy dư `%`

`a % b` là **số dư** khi chia `a` cho `b`. Với `b = 2` chỉ có hai kết quả: `0` (số **chẵn**) hoặc `1` (số **lẻ**). Ta gọi giá trị `x % 2` là **parity** (tính chẵn/lẻ) của `x`. Hai số có **cùng parity** thì `x % 2 == y % 2`; có **parity khác nhau** thì `x % 2 != y % 2`.

In [2]:
for x in [1, 2, 3, 4, 7, 10, 100]:
    print(x, "% 2 =", x % 2, "->", "lẻ" if x % 2 == 1 else "chẵn")

1 % 2 = 1 -> lẻ
2 % 2 = 0 -> chẵn
3 % 2 = 1 -> lẻ
4 % 2 = 0 -> chẵn
7 % 2 = 1 -> lẻ
10 % 2 = 0 -> chẵn
100 % 2 = 0 -> chẵn


### 3.2. Chỉ số (index) và cặp phần tử kề nhau

Phần tử kề nhau là `nums[i]` và `nums[i + 1]`. Mảng có `n` phần tử thì có **`n - 1` cặp kề nhau**: các cặp `(0,1), (1,2), ..., (n-2, n-1)`. Vì vậy chỉ số `i` chạy từ `0` đến `n - 2`, tức `range(n - 1)`. Nếu cho `i` chạy tới `n - 1` thì `nums[i + 1]` là `nums[n]` — **vượt chỉ số**.

In [3]:
nums = [4, 3, 1, 6]
n = len(nums)
print("n =", n, "-> số cặp kề nhau =", n - 1)
for i in range(n - 1):
    print(f"cặp {i}: ({nums[i]}, {nums[i + 1]})")

n = 4 -> số cặp kề nhau = 3
cặp 0: (4, 3)
cặp 1: (3, 1)
cặp 2: (1, 6)


### 3.3. `zip(nums, nums[1:])`: ghép mỗi phần tử với phần tử đứng sau nó

`nums[1:]` là mảng bỏ phần tử đầu. `zip(a, b)` ghép từng cặp song song và **dừng khi một bên hết**. Do đó `zip(nums, nums[1:])` cho đúng các cặp kề nhau `(nums[0], nums[1])`, `(nums[1], nums[2])`, ... mà không cần chỉ số.

In [4]:
nums = [2, 1, 4]
print(nums[1:])                          # [1, 4]
print(list(zip(nums, nums[1:])))         # [(2, 1), (1, 4)]
print(list(zip([1], [1][1:])))           # [] : mảng một phần tử không có cặp nào

[1, 4]
[(2, 1), (1, 4)]
[]


### 3.4. `all(...)` và `any(...)`

- `all(iterable)` trả về `True` nếu **mọi** phần tử đều đúng (và **`True` với dãy rỗng**).
- `any(iterable)` trả về `True` nếu **có ít nhất một** phần tử đúng (và `False` với dãy rỗng).

Hai hàm này **dừng sớm**: `all` dừng ngay khi gặp phần tử sai đầu tiên; `any` dừng ngay khi gặp phần tử đúng đầu tiên. Rất hợp cho bài này: "mọi cặp đều khác parity" = `all(...)`.

In [5]:
print(all([True, True, False]))   # False
print(all([]))                    # True  (rỗng: không có gì sai)
print(any([False, False, True]))  # True
print(any([]))                    # False
print(all(x % 2 == 1 for x in [1, 3, 5]))   # True: mọi số đều lẻ

False
True
True
False
True


### 3.5. Mẹo bit: `x & 1` và `a ^ b`

- `x & 1` lấy **bit cuối** của `x`: bằng `1` nếu `x` lẻ, `0` nếu `x` chẵn — giống `x % 2` nhưng viết bằng bit.
- `a ^ b` là **XOR**: bit cuối của `a ^ b` bằng `1` khi **bit cuối của `a` và `b` khác nhau**. Vậy `(a ^ b) & 1 == 1` nghĩa là **`a` và `b` khác parity**.

In [6]:
for a, b in [(2, 1), (3, 1), (4, 6), (5, 8)]:
    print(f"a={a} ({a:03b}), b={b} ({b:03b}) | a%2 != b%2: {a % 2 != b % 2} | (a ^ b) & 1: {(a ^ b) & 1}")

a=2 (010), b=1 (001) | a%2 != b%2: True | (a ^ b) & 1: 1
a=3 (011), b=1 (001) | a%2 != b%2: False | (a ^ b) & 1: 0
a=4 (100), b=6 (110) | a%2 != b%2: False | (a ^ b) & 1: 0
a=5 (101), b=8 (1000) | a%2 != b%2: True | (a ^ b) & 1: 1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [4, 3, 1, 6]`, ta **thay mỗi số bằng parity của nó** (`C` = chẵn, `L` = lẻ) rồi nhìn các cặp kề nhau:

| vị trí | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| số | 4 | 3 | 1 | 6 |
| parity | C | L | L | C |

Các cặp: `(C, L)` tốt, `(L, L)` **xấu**, `(L, C)` tốt. Chỉ cần **một** cặp xấu là mảng không đặc biệt.

Nhìn theo cách khác: mảng đặc biệt **phải có dạng xen kẽ** `C L C L ...` hoặc `L C L C ...`. Với `[2, 1, 4]` ta có `C L C` — xen kẽ hoàn hảo nên đúng.

In [7]:
nums = [4, 3, 1, 6]
parity = ["C" if x % 2 == 0 else "L" for x in nums]
print(parity)                                   # ['C', 'L', 'L', 'C']
for i in range(len(nums) - 1):
    ok = parity[i] != parity[i + 1]
    print(f"cặp ({nums[i]}, {nums[i + 1]}) -> {parity[i]}{parity[i + 1]} -> {'tốt' if ok else 'XẤU'}")

['C', 'L', 'L', 'C']
cặp (4, 3) -> CL -> tốt
cặp (3, 1) -> LL -> XẤU
cặp (1, 6) -> LC -> tốt


## 5. Cách 1 — Brute Force / baseline: xét mọi cặp `(i, j)` rồi chỉ giữ cặp kề nhau

**Ý tưởng.** Đọc đề sát từng chữ: "mọi cặp phần tử kề nhau". Ta lập hai vòng lặp để xét **mọi cặp** `(i, j)` với `i < j`, nhưng chỉ kiểm tra khi `j == i + 1` (kề nhau). Nếu cặp đó có cùng parity thì trả `False`. Cách này đúng nhưng lãng phí: ta duyệt cả những cặp xa nhau rồi bỏ đi.

In [8]:
def is_special_all_pairs(nums: List[int]) -> bool:
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            if j == i + 1:                              # chỉ xét cặp kề nhau
                if nums[i] % 2 == nums[j] % 2:          # cùng parity -> vi phạm
                    return False
    return True


for nums, expected in examples:
    print(nums, "->", is_special_all_pairs(nums), "(kỳ vọng", expected, ")")

[1] -> True (kỳ vọng True )
[2, 1, 4] -> True (kỳ vọng True )
[4, 3, 1, 6] -> False (kỳ vọng False )


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`. Hai vòng lồng nhau xét `~n²/2` cặp ⇒ **Time `O(n²)`**, **Space `O(1)`**. Với `n <= 100` vẫn tức thì, nhưng cách này không "đúng bản chất".

**Nút thắt:** ta chỉ cần `n - 1` cặp kề nhau mà lại duyệt `~n²/2` cặp. Vòng lặp trong thừa — với mỗi `i` chỉ có **một** `j` đáng xét là `i + 1`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bỏ vòng lặp trong.** Với mỗi `i` từ `0` đến `n - 2`, so sánh trực tiếp `nums[i]` với `nums[i + 1]`:

- Nếu `nums[i] % 2 == nums[i + 1] % 2` ⇒ cặp vi phạm ⇒ **trả `False` ngay**.
- Nếu duyệt hết mà không gặp cặp vi phạm ⇒ **trả `True`**.

Chú ý bất đối xứng: **`False` có thể kết luận sớm** (một bằng chứng là đủ), còn **`True` chỉ kết luận được sau khi đã xét hết mọi cặp**. Vì vậy `return True` phải nằm **ngoài** vòng lặp.

In [9]:
nums = [2, 1, 4]
for i in range(len(nums) - 1):
    a, b = nums[i], nums[i + 1]
    print(f"i={i}: ({a}, {b}) parity ({a % 2}, {b % 2}) -> {'cùng parity: vi phạm' if a % 2 == b % 2 else 'khác parity: ổn'}")
print("duyệt hết, không có vi phạm -> True")

i=0: (2, 1) parity (0, 1) -> khác parity: ổn
i=1: (1, 4) parity (1, 0) -> khác parity: ổn
duyệt hết, không có vi phạm -> True


## 7. Các cách tiếp theo

### Cách 2 — Một vòng lặp, so sánh cặp kề nhau bằng `%`

**Giải thích từng đoạn.**

1. `for i in range(len(nums) - 1)`: `i` đi qua mọi chỉ số có phần tử đứng sau.
2. `if nums[i] % 2 == nums[i + 1] % 2: return False`: gặp cặp cùng parity là dừng.
3. `return True`: không có cặp xấu nào (kể cả khi mảng chỉ có một phần tử, vòng lặp chạy `0` lần).

In [10]:
def is_special_loop(nums: List[int]) -> bool:
    for i in range(len(nums) - 1):
        if nums[i] % 2 == nums[i + 1] % 2:
            return False
    return True


for nums, expected in examples:
    print(nums, "->", is_special_loop(nums), "(kỳ vọng", expected, ")")

[1] -> True (kỳ vọng True )
[2, 1, 4] -> True (kỳ vọng True )
[4, 3, 1, 6] -> False (kỳ vọng False )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionLoop:
    def isArraySpecial(self, nums: List[int]) -> bool:
        for i in range(len(nums) - 1):
            if nums[i] % 2 == nums[i + 1] % 2:
                return False
        return True


print(SolutionLoop().isArraySpecial([4, 3, 1, 6]))      # False

False


**Độ phức tạp Cách 2:** **Time `O(n)`** (tối đa `n - 1` lần so sánh), **Space `O(1)`**.

### Cách 3 — `zip` + `all`

**Giải thích.** `zip(nums, nums[1:])` cho các cặp kề nhau `(a, b)`. Điều kiện "mọi cặp khác parity" viết thẳng thành `all(a % 2 != b % 2 for a, b in zip(nums, nums[1:]))`. Với mảng một phần tử, `zip` rỗng và `all([])` là `True` — đúng ý đề. (Lưu ý: `nums[1:]` tạo bản sao nên tốn `O(n)` bộ nhớ.)

In [12]:
def is_special_zip(nums: List[int]) -> bool:
    return all(a % 2 != b % 2 for a, b in zip(nums, nums[1:]))


for nums, expected in examples:
    print(nums, "->", is_special_zip(nums), "(kỳ vọng", expected, ")")

[1] -> True (kỳ vọng True )
[2, 1, 4] -> True (kỳ vọng True )
[4, 3, 1, 6] -> False (kỳ vọng False )


In [13]:
# Code nộp LeetCode cho Cách 3
class SolutionZip:
    def isArraySpecial(self, nums: List[int]) -> bool:
        return all(a % 2 != b % 2 for a, b in zip(nums, nums[1:]))


print(SolutionZip().isArraySpecial([2, 1, 4]))          # True

True


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** do `nums[1:]` (nếu dùng `itertools.pairwise` thì `O(1)`).

### Cách 4 — Mẹo bit: `(a ^ b) & 1`

**Giải thích.** Như mục 3.5, `(a ^ b) & 1 == 1` khi và chỉ khi `a`, `b` khác parity. Mảng đặc biệt khi **mọi** cặp có `(a ^ b) & 1 == 1`; ngược lại, cặp xấu là cặp có `(a ^ b) & 1 == 0`, tức `(a ^ b)` chẵn. Dùng `itertools.pairwise` (Python 3.10+) để không sao chép mảng.

In [14]:
def is_special_bit(nums: List[int]) -> bool:
    return all((a ^ b) & 1 for a, b in itertools.pairwise(nums))


for nums, expected in examples:
    print(nums, "->", is_special_bit(nums), "(kỳ vọng", expected, ")")

[1] -> True (kỳ vọng True )
[2, 1, 4] -> True (kỳ vọng True )
[4, 3, 1, 6] -> False (kỳ vọng False )


In [15]:
# Code nộp LeetCode cho Cách 4 (nếu máy chấm không có pairwise, dùng zip(nums, nums[1:]))
class SolutionBit:
    def isArraySpecial(self, nums: List[int]) -> bool:
        for i in range(len(nums) - 1):
            if (nums[i] ^ nums[i + 1]) & 1 == 0:        # bit cuối giống nhau -> cùng parity
                return False
        return True


print(SolutionBit().isArraySpecial([4, 3, 1, 6]))       # False

False


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`** (bản vòng lặp).

### Cách 5 — Mẫu chẵn/lẻ xen kẽ: parity của `nums[i]` phải bằng `(nums[0] + i) % 2`

**Giải thích.** Nếu mảng đặc biệt thì parity **đổi dấu mỗi bước**: phần tử `0` có parity `p = nums[0] % 2`, phần tử `1` có parity `1 - p`, phần tử `2` lại `p`, ... Tổng quát parity của `nums[i]` phải là `(p + i) % 2`. Ta chỉ cần kiểm tra **mỗi phần tử một lần** với "parity mong đợi" thay vì so từng cặp. Hai cách nghĩ này tương đương (quy nạp theo `i`).

In [16]:
def is_special_pattern(nums: List[int]) -> bool:
    p = nums[0] % 2                       # parity của phần tử đầu
    for i, x in enumerate(nums):
        if x % 2 != (p + i) % 2:          # parity mong đợi tại vị trí i
            return False
    return True


for nums, expected in examples:
    print(nums, "->", is_special_pattern(nums), "(kỳ vọng", expected, ")")

[1] -> True (kỳ vọng True )
[2, 1, 4] -> True (kỳ vọng True )
[4, 3, 1, 6] -> False (kỳ vọng False )


In [17]:
# Code nộp LeetCode cho Cách 5
class SolutionPattern:
    def isArraySpecial(self, nums: List[int]) -> bool:
        p = nums[0] % 2
        for i, x in enumerate(nums):
            if x % 2 != (p + i) % 2:
                return False
        return True


print(SolutionPattern().isArraySpecial([2, 1, 4]))      # True

True


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`**. (Đề đảm bảo `n >= 1`, nên `nums[0]` luôn tồn tại.)

## 8. TRACE từng bước

Ta trace Cách 2 (một vòng lặp) trên `nums = [4, 3, 1, 6]` (kết quả `false`) và `nums = [2, 1, 4]` (kết quả `true`), rồi trace Cách 5 (mẫu xen kẽ).

In [18]:
def trace_loop(nums: List[int]) -> bool:
    print(f"nums = {nums}")
    for i in range(len(nums) - 1):
        a, b = nums[i], nums[i + 1]
        same = a % 2 == b % 2
        print(f"  i={i}: a={a} (parity {a % 2}), b={b} (parity {b % 2}) -> {'CÙNG parity => return False' if same else 'khác parity, tiếp tục'}")
        if same:
            return False
    print("  hết vòng lặp, không có vi phạm => return True")
    return True


assert trace_loop([4, 3, 1, 6]) is False
assert trace_loop([2, 1, 4]) is True
assert trace_loop([1]) is True

nums = [4, 3, 1, 6]
  i=0: a=4 (parity 0), b=3 (parity 1) -> khác parity, tiếp tục
  i=1: a=3 (parity 1), b=1 (parity 1) -> CÙNG parity => return False
nums = [2, 1, 4]
  i=0: a=2 (parity 0), b=1 (parity 1) -> khác parity, tiếp tục
  i=1: a=1 (parity 1), b=4 (parity 0) -> khác parity, tiếp tục
  hết vòng lặp, không có vi phạm => return True
nums = [1]
  hết vòng lặp, không có vi phạm => return True


In [19]:
def trace_pattern(nums: List[int]) -> bool:
    p = nums[0] % 2
    print(f"nums = {nums}, parity phần tử đầu p = {p}")
    for i, x in enumerate(nums):
        expected = (p + i) % 2
        print(f"  i={i}: x={x}, parity thật = {x % 2}, parity mong đợi (p + i) % 2 = {expected}")
        if x % 2 != expected:
            print("  -> lệch => return False")
            return False
    print("  -> khớp hết => return True")
    return True


assert trace_pattern([4, 3, 1, 6]) is False
assert trace_pattern([2, 1, 4]) is True

nums = [4, 3, 1, 6], parity phần tử đầu p = 0
  i=0: x=4, parity thật = 0, parity mong đợi (p + i) % 2 = 0
  i=1: x=3, parity thật = 1, parity mong đợi (p + i) % 2 = 1
  i=2: x=1, parity thật = 1, parity mong đợi (p + i) % 2 = 0
  -> lệch => return False
nums = [2, 1, 4], parity phần tử đầu p = 0
  i=0: x=2, parity thật = 0, parity mong đợi (p + i) % 2 = 0
  i=1: x=1, parity thật = 1, parity mong đợi (p + i) % 2 = 1
  i=2: x=4, parity thật = 0, parity mong đợi (p + i) % 2 = 0
  -> khớp hết => return True


Bảng trace Cách 2 với `nums = [4, 3, 1, 6]`:

| i | `nums[i]` | `nums[i+1]` | parity (a, b) | khác nhau? | hành động |
|---|---|---|---|---|---|
| 0 | 4 | 3 | (0, 1) | có | tiếp tục |
| 1 | 3 | 1 | (1, 1) | **không** | **return False** |

Vòng lặp dừng ở `i = 1`; cặp `(1, 6)` không cần xét nữa. Đáp án `false`.

Bảng trace Cách 2 với `nums = [2, 1, 4]`:

| i | `nums[i]` | `nums[i+1]` | parity (a, b) | khác nhau? | hành động |
|---|---|---|---|---|---|
| 0 | 2 | 1 | (0, 1) | có | tiếp tục |
| 1 | 1 | 4 | (1, 0) | có | tiếp tục |

Hết vòng lặp ⇒ `return True`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (một vòng lặp, dừng sớm): rõ ràng, `O(n)` thời gian, `O(1)` bộ nhớ.

In [20]:
class Solution:
    def isArraySpecial(self, nums: List[int]) -> bool:
        for i in range(len(nums) - 1):                     # mọi cặp kề nhau (i, i + 1)
            if nums[i] % 2 == nums[i + 1] % 2:             # cùng chẵn hoặc cùng lẻ
                return False                               # một cặp xấu là đủ
        return True                                        # không có cặp xấu nào


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.isArraySpecial(nums), "(kỳ vọng", expected, ")")

[1] -> True (kỳ vọng True )
[2, 1, 4] -> True (kỳ vọng True )
[4, 3, 1, 6] -> False (kỳ vọng False )


## 10. Vì sao thuật toán đúng

- **Định nghĩa đã là một điều kiện "với mọi":** mảng đặc biệt ⇔ với mọi `i` trong `[0, n-2]`, `nums[i]` và `nums[i+1]` khác parity. Phủ định của "với mọi" là "tồn tại": mảng **không** đặc biệt ⇔ **tồn tại** `i` mà hai phần tử kề nhau cùng parity.
- **`False` đúng:** nếu vòng lặp gặp `i` có cùng parity thì ta vừa tìm được "tồn tại" đó, nên trả `False` là đúng, không cần xét thêm.
- **`True` đúng:** nếu vòng lặp chạy hết mà không có `i` nào vi phạm thì mọi cặp kề nhau đều khác parity ⇒ mảng đặc biệt.
- **Mảng một phần tử:** `range(0)` rỗng, không có cặp nào, ta trả `True` — khớp Ví dụ 1.
- **Khớp với mẫu xen kẽ (Cách 5):** nếu hai phần tử kề nhau luôn khác parity thì parity đổi `0 ↔ 1` mỗi bước, nên parity của `nums[i]` là `(nums[0] % 2 + i) % 2`; và ngược lại, nếu parity khớp công thức đó thì hai phần tử kề nhau chênh nhau đúng `1` đơn vị trong `(p + i) % 2`, tức khác nhau.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (`n <= 100`).

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp `(i, j)` lọc `j == i + 1` | `O(n²)` | `O(1)` |
| 2. Một vòng lặp so sánh `%` | `O(n)` | `O(1)` |
| 3. `zip` + `all` | `O(n)` | `O(n)` (do `nums[1:]`) |
| 4. Mẹo bit `(a ^ b) & 1` | `O(n)` | `O(1)` |
| 5. Mẫu xen kẽ | `O(n)` | `O(1)` |

Không thể nhanh hơn `O(n)` trong trường hợp xấu nhất vì để trả `True` ta phải nhìn **mọi** phần tử ít nhất một lần.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [21]:
sol = Solution()
impls = {
    "all_pairs": is_special_all_pairs,
    "loop": is_special_loop,
    "zip": is_special_zip,
    "bit": is_special_bit,
    "pattern": is_special_pattern,
    "leetcode": sol.isArraySpecial,
}


def check_all(nums, expected, label):
    for key, fn in impls.items():
        actual = fn(list(nums))
        assert actual == expected, (label, key, nums, actual, expected)
    print("PASS", label)


for idx, (nums, expected) in enumerate(examples, 1):
    check_all(nums, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [22]:
check_all([5], True, "một phần tử lẻ")
check_all([100], True, "một phần tử chẵn")
check_all([1, 2], True, "hai phần tử khác parity")
check_all([2, 1], True, "hai phần tử khác parity (chẵn trước)")
check_all([2, 4], False, "hai phần tử cùng chẵn")
check_all([3, 3], False, "hai phần tử bằng nhau (cùng lẻ)")
check_all([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], True, "1..10 xen kẽ lẻ chẵn")
check_all([2, 3, 4, 5, 6, 7, 8], True, "bắt đầu bằng chẵn")
check_all([1, 2, 3, 4, 5, 6, 7, 8, 9, 9], False, "chỉ cặp cuối vi phạm")
check_all([2, 2, 1, 2, 1], False, "chỉ cặp đầu vi phạm")
check_all([1, 2, 1, 1, 2], False, "vi phạm ở giữa")
check_all([100] * 100, False, "100 phần tử cùng chẵn")
check_all([1, 100] * 50, True, "100 phần tử xen kẽ")

PASS một phần tử lẻ
PASS một phần tử chẵn
PASS hai phần tử khác parity
PASS hai phần tử khác parity (chẵn trước)
PASS hai phần tử cùng chẵn
PASS hai phần tử bằng nhau (cùng lẻ)
PASS 1..10 xen kẽ lẻ chẵn
PASS bắt đầu bằng chẵn
PASS chỉ cặp cuối vi phạm
PASS chỉ cặp đầu vi phạm
PASS vi phạm ở giữa
PASS 100 phần tử cùng chẵn
PASS 100 phần tử xen kẽ


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **so sánh giá trị** `nums[i] != nums[i+1]` thay vì parity; (b) **đếm số chẵn và số lẻ bằng nhau** thay vì xét cặp kề nhau.

In [23]:
def wrong_compare_value(nums):
    return all(a != b for a, b in zip(nums, nums[1:]))              # SAI: so sánh giá trị


def wrong_count_equal(nums):
    evens = sum(1 for x in nums if x % 2 == 0)
    return abs(evens - (len(nums) - evens)) <= 1                    # SAI: chỉ đếm, không xét thứ tự


print(wrong_compare_value([2, 4]), "!=", is_special_loop([2, 4]))        # [2,4]: khác giá trị nhưng cùng chẵn
print(wrong_count_equal([2, 4, 1, 3]), "!=", is_special_loop([2, 4, 1, 3]))   # 2 chẵn 2 lẻ nhưng không xen kẽ
assert wrong_compare_value([2, 4]) != is_special_loop([2, 4])
assert wrong_count_equal([2, 4, 1, 3]) != is_special_loop([2, 4, 1, 3])

True != False
True != False


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn

Đầu vào hợp lệ: độ dài `1..100`, giá trị `1..100`. Ta kiểm tra: (1) **ngẫu nhiên** hoàn toàn; (2) **mảng đặc biệt được dựng sẵn** (chọn parity xen kẽ rồi chọn số ngẫu nhiên đúng parity) phải ra `True`; (3) **chèn một cặp xấu** vào mảng đó phải ra `False`; (4) **vét cạn** mọi mảng độ dài `1..10` tạo từ `{1, 2}` (chỉ parity mới quan trọng) và đối chiếu với một "oracle" khác dựa trên chuỗi parity.

In [24]:
def oracle(nums):
    """Chuyển thành chuỗi parity '0'/'1' rồi tìm '00' hoặc '11'."""
    s = "".join(str(x % 2) for x in nums)
    return "00" not in s and "11" not in s


rng = random.Random(3151)
for trial in range(3000):
    n = rng.randint(1, 100)
    nums = [rng.randint(1, 100) for _ in range(n)]
    ref = oracle(nums)
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
print("PASS 3000 mảng ngẫu nhiên")

# mảng đặc biệt dựng sẵn
for trial in range(2000):
    n = rng.randint(1, 100)
    start = rng.randint(0, 1)
    nums = []
    for i in range(n):
        want = (start + i) % 2
        nums.append(rng.choice([v for v in range(1, 101) if v % 2 == want]))
    for key, fn in impls.items():
        assert fn(list(nums)) is True, (key, nums)
    if n >= 2:
        k = rng.randrange(n - 1)                       # làm hỏng cặp (k, k+1): đổi parity của nums[k+1]
        bad = list(nums)
        bad[k + 1] = nums[k + 1] + 1 if nums[k + 1] < 100 else nums[k + 1] - 1
        for key, fn in impls.items():
            assert fn(list(bad)) is False, (key, bad)          # đổi parity nums[k+1] => cặp (k, k+1) cùng parity
            assert fn(list(bad)) == oracle(bad), (key, bad)
print("PASS 2000 mảng đặc biệt dựng sẵn + biến thể bị phá")

count = 0
for n in range(1, 11):
    for combo in itertools.product([1, 2], repeat=n):
        ref = oracle(combo)
        for key, fn in impls.items():
            assert fn(list(combo)) == ref, (key, combo)
        count += 1
print("PASS vét cạn", count, "mảng nhị phân độ dài 1..10")

PASS 3000 mảng ngẫu nhiên


PASS 2000 mảng đặc biệt dựng sẵn + biến thể bị phá
PASS vét cạn 2046 mảng nhị phân độ dài 1..10


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Duyệt `range(len(nums))` rồi đọc `nums[i + 1]` (IndexError)

In [25]:
def sai_index_error(nums):
    for i in range(len(nums)):                      # SAI: i chạy tới n - 1
        if nums[i] % 2 == nums[i + 1] % 2:          # i = n - 1 -> nums[n] vượt chỉ số
            return False
    return True


try:
    sai_index_error([2, 1, 4])
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().isArraySpecial([2, 1, 4]))

IndexError: list index out of range
đúng: True


**Vì sao sai:** ở vòng cuối `i = n - 1`, `nums[i + 1]` không tồn tại. **Cách sửa:** `range(len(nums) - 1)` (hoặc `zip(nums, nums[1:])`).

### Lỗi 2: `return True` quá sớm

In [26]:
def sai_return_true_som(nums):
    for i in range(len(nums) - 1):
        if nums[i] % 2 != nums[i + 1] % 2:
            return True                              # SAI: một cặp tốt chưa chứng minh được gì
    return False


print("sai:", sai_return_true_som([2, 1, 3]), "| đúng:", Solution().isArraySpecial([2, 1, 3]))
print("sai:", sai_return_true_som([2, 4]), "| đúng:", Solution().isArraySpecial([2, 4]))
print("sai:", sai_return_true_som([1]), "| đúng:", Solution().isArraySpecial([1]))

sai: True | đúng: False
sai: False | đúng: False
sai: False | đúng: True


**Vì sao sai:** điều kiện đặt ra là **mọi** cặp đều tốt, nên **một** cặp tốt không đủ để kết luận `True`; ngược lại **một** cặp xấu mới đủ để kết luận `False`. Mảng `[2, 1, 3]` có cặp đầu tốt nhưng cặp sau `(1, 3)` xấu. Hàm sai còn trả `False` cho `[1]` trong khi đáp án là `True`. **Cách sửa:** trả `False` khi gặp cặp xấu, và `True` chỉ ở cuối.

### Lỗi 3: So sánh giá trị thay vì parity

In [27]:
def sai_so_gia_tri(nums):
    for i in range(len(nums) - 1):
        if nums[i] == nums[i + 1]:                   # SAI: chỉ bắt được hai số BẰNG nhau
            return False
    return True


print("sai:", sai_so_gia_tri([4, 3, 1, 6]), "| đúng:", Solution().isArraySpecial([4, 3, 1, 6]))
print("sai:", sai_so_gia_tri([2, 4]), "| đúng:", Solution().isArraySpecial([2, 4]))

sai: True | đúng: False
sai: True | đúng: False


**Vì sao sai:** `3` và `1` khác giá trị nhưng cùng lẻ. Đề so **parity**, không so giá trị. **Cách sửa:** so `nums[i] % 2` với `nums[i + 1] % 2`.

### Lỗi 4: Quên mảng một phần tử và dùng `nums[1]`

In [28]:
def sai_quen_mot_phan_tu(nums):
    if nums[0] % 2 == nums[1] % 2:                   # SAI: mảng [1] không có nums[1]
        return False
    return all(a % 2 != b % 2 for a, b in zip(nums, nums[1:]))


try:
    sai_quen_mot_phan_tu([1])
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().isArraySpecial([1]))

IndexError: list index out of range
đúng: True


**Vì sao sai:** đề cho phép `nums.length = 1` (không có cặp). Truy cập cố định `nums[1]` làm hỏng ca này. **Cách sửa:** để vòng lặp/`zip` tự xử lý (không có cặp ⇒ không có vi phạm ⇒ `True`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Mọi cặp phần tử kề nhau thỏa điều kiện**" ⇒ một vòng lặp `for i in range(n - 1)` hoặc `all(... for a, b in zip(nums, nums[1:]))`.
- Điều kiện "với mọi" ⇒ **tìm phản ví dụ và dừng sớm**; duyệt hết mà không thấy ⇒ `True`.
- Chẵn/lẻ ⇒ `x % 2` hoặc `x & 1`. "Hai số khác parity" ⇒ `(a + b) % 2 == 1` hoặc `(a ^ b) & 1`.
- Với `n` phần tử luôn có `n - 1` cặp kề nhau: viết `range(n - 1)` ngay từ đầu để khỏi quên.
- Mảng "xen kẽ" (alternating) là một họ bài: *Alternating Subarrays*, *Zigzag*, *Wiggle*. Hãy nghĩ tới "so sánh phần tử với phần tử liền kề".
- Luôn thử ca `n = 1` và ca hai phần tử.

## 15. Biến thể của bài

1. **Special Array II (3152):** trả lời nhiều truy vấn `[l, r]` — mảng con `nums[l..r]` có đặc biệt không? (dùng mảng cộng dồn số cặp xấu).
2. **Đếm số cặp xấu:** có bao nhiêu cặp kề nhau cùng parity?
3. **Độ dài đoạn con đặc biệt dài nhất** (chẵn/lẻ xen kẽ liên tục dài nhất).
4. **Đổi bao nhiêu phần tử tối thiểu** (cộng/trừ 1) để mảng trở thành đặc biệt?
5. Bài gần gũi: **Monotonic Array (896)**, **Check if Array Is Sorted and Rotated (1752)**, **Longest Turbulent Subarray (978)**.

In [29]:
def count_bad_pairs(nums):
    return sum(1 for a, b in zip(nums, nums[1:]) if a % 2 == b % 2)


def longest_special_subarray(nums):
    best = cur = 1
    for i in range(1, len(nums)):
        cur = cur + 1 if nums[i] % 2 != nums[i - 1] % 2 else 1
        best = max(best, cur)
    return best


def min_changes_to_special(nums):
    """Đổi parity một phần tử (cộng 1) tốn 1 bước: chọn mẫu bắt đầu chẵn hoặc lẻ, lấy mẫu ít lệch hơn."""
    mismatch_a = sum(1 for i, x in enumerate(nums) if x % 2 != i % 2)          # mẫu: vị trí chẵn có parity 0
    return min(mismatch_a, len(nums) - mismatch_a)


print(count_bad_pairs([4, 3, 1, 6]))                # 1
print(longest_special_subarray([4, 3, 1, 6, 5]))    # 3 : 1,6,5
print(min_changes_to_special([2, 4, 6]))            # 1 : đổi số giữa
assert count_bad_pairs([4, 3, 1, 6]) == 1
assert count_bad_pairs([1]) == 0
assert longest_special_subarray([4, 3, 1, 6, 5]) == 3
assert longest_special_subarray([2, 4, 6]) == 1
assert min_changes_to_special([2, 4, 6]) == 1
assert min_changes_to_special([2, 1, 4]) == 0

1
3
1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp, lọc `j == i + 1` | đọc đề từng chữ | `O(n²)` | `O(1)` | chỉ để hiểu đề, không nên nộp |
| 2. Một vòng lặp `%` | so `nums[i]` với `nums[i + 1]`, dừng sớm | `O(n)` | `O(1)` | **nên nộp**: rõ ràng nhất |
| 3. `zip` + `all` | ghép cặp kề nhau, kiểm tra mọi cặp | `O(n)` | `O(n)` | viết ngắn, quen cú pháp Python |
| 4. Bit `(a ^ b) & 1` | bit cuối khác nhau ⇔ khác parity | `O(n)` | `O(1)` | luyện tư duy bit |
| 5. Mẫu xen kẽ | parity `i` phải bằng `(nums[0] + i) % 2` | `O(n)` | `O(1)` | khi cần "mẫu mong đợi" (bài tương tự) |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `7 % 2`, `10 & 1`, `(6 ^ 9) & 1`, `list(zip([1, 2, 3], [2, 3]))`, `all([])`.

**Bài 2 (tính tay).** Với `nums = [3, 8, 5, 10, 7]` và `nums = [3, 8, 5, 5, 10]`: viết chuỗi parity (`C`/`L`) và cho biết mảng nào đặc biệt.

**Bài 3 (điền TODO).** Hoàn thành `is_special_todo(nums)` theo Cách 2.

In [30]:
def is_special_todo(nums):
    # TODO 1: duyệt i từ 0 đến len(nums) - 2
    # TODO 2: nếu nums[i] và nums[i + 1] cùng parity thì return False
    # TODO 3: sau vòng lặp return True
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `is_special_pairwise(nums)` dùng `itertools.pairwise` và `any(...)` (tìm một cặp xấu) thay vì `all(...)`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `first_bad_index(nums)` trả về chỉ số `i` đầu tiên mà `nums[i]`, `nums[i + 1]` cùng parity, hoặc `-1` nếu mảng đặc biệt. (b) Viết `make_special(nums)` trả về một mảng đặc biệt **cùng độ dài** bằng cách cộng `1` vào những phần tử cần thiết, đi từ trái sang phải (giữ nguyên phần tử đầu).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `7 % 2 = 1`; `10 & 1 = 0`; `6 ^ 9 = 15` nên `(6 ^ 9) & 1 = 1`; `zip` dừng khi bên ngắn hết nên ra `[(1, 2), (2, 3)]`; `all([])` là `True`. Bài 3: vòng lặp `for i in range(len(nums) - 1)`.
- **Mức 2 (vừa):** Bài 2: `[3, 8, 5, 10, 7]` ⇒ `L C L C L` (xen kẽ, đặc biệt); `[3, 8, 5, 5, 10]` ⇒ `L C L L C` (cặp `(5, 5)` vi phạm). Bài 4: `not any(a % 2 == b % 2 for a, b in itertools.pairwise(nums))`.
- **Mức 3 (sâu):** Bài 5(a): vòng lặp `range(len(nums) - 1)`, gặp cặp cùng parity thì `return i`, hết vòng lặp `return -1`. 5(b): giữ `res = [nums[0]]`; với mỗi phần tử kế tiếp `x`, nếu `x % 2 == res[-1] % 2` thì thêm `x + 1`, ngược lại thêm `x`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert 7 % 2 == 1
assert 10 & 1 == 0
assert (6 ^ 9) & 1 == 1
assert list(zip([1, 2, 3], [2, 3])) == [(1, 2), (2, 3)]
assert all([]) is True

# Bài 2
def parity_string(nums):
    return "".join("C" if x % 2 == 0 else "L" for x in nums)


assert parity_string([3, 8, 5, 10, 7]) == "LCLCL"
assert parity_string([3, 8, 5, 5, 10]) == "LCLLC"
assert Solution().isArraySpecial([3, 8, 5, 10, 7]) is True
assert Solution().isArraySpecial([3, 8, 5, 5, 10]) is False


# Bài 3
def is_special_todo_ans(nums):
    for i in range(len(nums) - 1):
        if nums[i] % 2 == nums[i + 1] % 2:
            return False
    return True


for nums, expected in examples:
    assert is_special_todo_ans(nums) == expected


# Bài 4
def is_special_pairwise(nums):
    return not any(a % 2 == b % 2 for a, b in itertools.pairwise(nums))


for nums, expected in examples:
    assert is_special_pairwise(nums) == expected


# Bài 5(a)
def first_bad_index(nums):
    for i in range(len(nums) - 1):
        if nums[i] % 2 == nums[i + 1] % 2:
            return i
    return -1


assert first_bad_index([4, 3, 1, 6]) == 1
assert first_bad_index([2, 1, 4]) == -1
assert first_bad_index([1]) == -1


# Bài 5(b)
def make_special(nums):
    res = [nums[0]]
    for x in nums[1:]:
        res.append(x + 1 if x % 2 == res[-1] % 2 else x)
    return res


assert make_special([4, 3, 1, 6]) == [4, 3, 2, 7]
assert Solution().isArraySpecial(make_special([4, 3, 1, 6]))
rng2 = random.Random(1)
for _ in range(500):
    arr = [rng2.randint(1, 100) for _ in range(rng2.randint(1, 30))]
    fixed = make_special(arr)
    assert len(fixed) == len(arr) and Solution().isArraySpecial(fixed)
    assert fixed[0] == arr[0]
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mọi cặp kề nhau khác parity)

```python
class Solution:
    def isArraySpecial(self, nums: List[int]) -> bool:
        for i in range(len(nums) - 1):
            if nums[i] % 2 == nums[i + 1] % 2:
                return False
        return True
```

**Checklist**

- [ ] Vòng lặp là `range(len(nums) - 1)` (không bị `IndexError` ở `nums[i + 1]`)?
- [ ] So sánh **parity** (`% 2`), không so giá trị?
- [ ] `return False` ngay khi gặp cặp xấu, còn `return True` nằm **ngoài** vòng lặp?
- [ ] Mảng một phần tử trả `True`?
- [ ] Đã thử `[1]` (`true`), `[2,1,4]` (`true`), `[4,3,1,6]` (`false`), và hai phần tử cùng parity `[2,4]` (`false`)?
- [ ] Skeleton đúng `class Solution` / `def isArraySpecial(self, nums: List[int]) -> bool`?